# Financial Knowledge Graph — End-to-End GraphRAG Pipeline
**FinBERT-GNN Extraction → Hierarchical Graph → Mistral Retrieval**

```
Bloomberg Financial News 120K
  │
  ▼  Preprocessing         (ticker anchoring, normalise, chunk)
  ▼  FinBERT Encoder        (ProsusAI/finbert → token hidden states H ∈ ℝⁿˣᵈ)
  ▼  EARE Module            (entity-aware gating → Ĥ)
  ▼  SSHG Construction      (dep arcs + cosine co-occurrence → hybrid graph)
  ▼  GNN Propagation        (2-layer message passing → H_gnn)
  ▼  BIO Entity Classifier  (MLP → BIO tags → entity spans)
  ▼  NEL                    (ticker → LEI/GID alias → fuzzy → LOCAL fingerprint)
  ▼  Temporal Layer         (PIT date extraction + validity enforcement)
  ▼  Relation Extraction    [Bilinear head + pattern prior]  ← primary
                            [Ollama LLM joint]               ← supplementary
  ▼  Provenance Enforcement (source_chunk_id + raw_sentence — hard invariant)
  ▼  Graph Builder          (append-only, versioned, canonical_id-keyed)
  ▼  Community Detection    (Leiden — 3-level hierarchy: L0 / L1 / L2)
  ▼  Community Summaries    (Ollama, grounded in graph edges only)
  ▼  Dual Retrieval         (Local KG-RAG + Global hierarchical + Hybrid LLM)
  ▼  Conversational RAG     (Mistral-Nemo, multi-turn, query rewrite)
  ▼  RAG Triad Evaluation   (Faithfulness · Answer Relevance · Context Precision)
```

**Why this stack?**
| Decision | Rationale |
|----------|-----------|
| FinBERT over spaCy/RoBERTa | Pre-trained on financial corpora; paper ablation shows best financial NER |
| EARE + SSHG + GNN | Structural sentence graph enriches entity boundary detection in dense financial text |
| LEI/GID canonical IDs | Legal Entity Identifiers are the financial industry standard; prevents alias explosion |
| Bilinear head primary RE | Domain-specific taxonomy + FinBERT embeddings; outperforms generic SVO patterns |
| Ollama LLM as supplementary | Catches novel/zero-shot relations the taxonomy doesn't cover |
| Leiden community detection | More recent (2024 Microsoft GraphRAG) than Louvain; better resolution |
| Temporal PIT enforcement | Financial facts are time-bounded; point-in-time validity is non-negotiable |

**Dataset**: Bloomberg Financial News 120K · **Team**: Malak Kably, Safae Hajjout

## 1. Environment Setup

Run once. All packages required for the full pipeline.

In [1]:
# Core
!pip install spacy rapidfuzz networkx python-dateutil requests datasets -q
!python -m spacy download en_core_web_sm -q

# FinBERT / GNN stack
!pip install transformers torch tqdm scipy scikit-learn -q

# Community detection
!pip install leidenalg python-igraph -q   # falls back to networkx if unavailable

# Visualisation
!pip install matplotlib pandas numpy -q

print("All dependencies installed.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
All dependencies installed.


## 2. Global Schema

Global Contract (Schema)

Every entity must have a `canonical_id`.  
Every relation must carry `provenance` (source_chunk_id + raw_sentence) and a `time` dict.  
Community nodes are derived **only** from graph edges — no external knowledge injection.


In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict, Any
import json, uuid, hashlib, re, os, time, requests, math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import spacy
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42)
np.random.seed(42)
# Force CPU - set FORCE_CPU=False to auto-detect GPU if your CUDA build matches your driver
FORCE_CPU = True
DEVICE = torch.device("cpu") if FORCE_CPU else torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


# -- Shared dataclasses ---------------------------------------------------------
@dataclass
class Entity:
    text: str
    type: str
    span: Tuple[int, int]

@dataclass
class Link:
    entity: str
    canonical_id: str
    confidence: float
    valid_time: Dict[str, Optional[str]]

@dataclass
class Provenance:
    source_chunk_id: str
    raw_sentence: str

@dataclass
class Relation:
    subj: str
    rel: str
    obj: str
    time: Dict[str, Optional[str]]
    provenance: Provenance
    confidence: float = 1.0

@dataclass
class Chunk:
    doc_id: str
    chunk_id: str
    text: str
    entities: List[Entity] = field(default_factory=list)
    links: List[Link] = field(default_factory=list)
    relations: List[Relation] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "doc_id": self.doc_id, "chunk_id": self.chunk_id, "text": self.text,
            "entities": [{"text": e.text, "type": e.type, "span": list(e.span)} for e in self.entities],
            "links": [{"entity": l.entity, "canonical_id": l.canonical_id,
                       "confidence": l.confidence, "valid_time": l.valid_time} for l in self.links],
            "relations": [{"subj": r.subj, "rel": r.rel, "obj": r.obj, "time": r.time,
                           "provenance": {"source_chunk_id": r.provenance.source_chunk_id,
                                         "raw_sentence": r.provenance.raw_sentence},
                           "confidence": r.confidence} for r in self.relations],
        }

@dataclass
class CommunityNode:
    cluster_id: str
    level: int
    summary: str
    entities: List[str] = field(default_factory=list)
    key_relations: List[Dict[str, Any]] = field(default_factory=list)


# -- Pipeline configuration -----------------------------------------------------
CFG = {
    "plm_name"           : "ProsusAI/finbert",
    "spacy_model"        : "en_core_web_sm",
    "max_seq_len"        : 256,
    "batch_size"         : 16,
    "window_size"        : 5,
    "lambda_syn"         : 1.0,
    "lambda_sem"         : 0.5,
    "gnn_hidden"         : 256,
    "gnn_layers"         : 2,
    "relation_threshold" : 0.30,
    "alpha_blend"        : 0.60,
    "ollama_url"         : "http://localhost:11434",
    "n_articles"         : 5000,
}

FINANCIAL_ENT_TYPES = {
    "ORG", "PERSON", "GPE", "MONEY", "DATE", "PERCENT",
    "QUANTITY", "CARDINAL", "FAC", "PRODUCT", "EVENT", "NORP",
}
TICKER_PATTERN = re.compile(r'\(([A-Z]{1,5}(?::[A-Z]{2})?)\)|\$([A-Z]{1,5})')
print("Schema and configuration loaded.")

/home/aries/projects/academic/GenAIProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Schema and configuration loaded.


## 3. Dataset Loading — Bloomberg Financial News 120K

In [3]:
print("Loading Bloomberg Financial News 120K...")
dataset  = load_dataset("XJCEO/bloomberg_financial_news_120k", split="train")
df       = dataset.to_pandas()
TEXT_COL = "Article" if "Article" in df.columns else df.select_dtypes("object").columns[0]

df = df[df[TEXT_COL].apply(lambda x: isinstance(x, str) and len(x.strip()) > 30)].reset_index(drop=True)
df = df.head(CFG["n_articles"]).copy()
df["article_id"] = df.index

print(f"Working subset : {len(df):,} articles")
print(f"Text column    : '{TEXT_COL}'")
print(df[[TEXT_COL]].head(2).to_string())

Loading Bloomberg Financial News 120K...
Working subset : 5,000 articles
Text column    : 'Article'
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

## 4. Preprocessing Layer

Ticker anchoring, text normalisation, and fixed-size chunking. Ticker anchors (e.g. `GS:US → Goldman Sachs`) are fed into the NEL stage.

In [4]:
TICKER_PATTERN = re.compile(r'\(([A-Z]{1,5}(?::[A-Z]{2})?)\)|\$([A-Z]{1,5})')


class Preprocessor:
    """Normalize text, extract ticker anchors, chunk into fixed-size windows."""

    def __init__(self, chunk_size: int = 512, ticker_linking: bool = True):
        self.chunk_size = chunk_size
        self.ticker_linking = ticker_linking
        self._ticker_kv: Dict[str, str] = {}

    def extract_tickers(self, text: str) -> List[Tuple[str, int, int]]:
        return [(m.group(1) or m.group(2), m.start(), m.end())
                for m in TICKER_PATTERN.finditer(text)]

    def normalize(self, text: str) -> str:
        return re.sub(r'\s+', ' ', text).strip()

    def chunk(self, doc_id: str, text: str) -> List[Chunk]:
        text = self.normalize(text)
        words = text.split()
        chunks = []
        for i in range(0, max(len(words), 1), self.chunk_size):
            chunk_text = ' '.join(words[i:i + self.chunk_size])
            chunk_id = hashlib.md5(f'{doc_id}:{i}'.encode()).hexdigest()[:12]
            chunks.append(Chunk(doc_id=doc_id, chunk_id=chunk_id, text=chunk_text))
        return chunks

    def anchor_tickers(self, chunk: Chunk) -> Dict[str, str]:
        if not self.ticker_linking:
            return {}
        anchors = {}
        for ticker, _, _ in self.extract_tickers(chunk.text):
            self._ticker_kv.setdefault(ticker, f'TICKER:{ticker}')
            anchors[ticker] = self._ticker_kv[ticker]
        return anchors

    def process(self, doc_id: str, text: str) -> Tuple[List[Chunk], Dict[str, str]]:
        chunks = self.chunk(doc_id, text)
        all_anchors: Dict[str, str] = {}
        for chunk in chunks:
            all_anchors.update(self.anchor_tickers(chunk))
        return chunks, all_anchors


_pp = Preprocessor(chunk_size=50)
_chunks, _anchors = _pp.process('test', 'Goldman Sachs (GS:US) acquired Marcus in 2022.')
print(f'chunks={len(_chunks)}, anchors={_anchors}')


chunks=1, anchors={'GS:US': 'TICKER:GS:US'}


## 5. FinBERT Encoder (PLM — Paper §3.2)

`ProsusAI/finbert` is pre-trained on financial corpora (Reuters, financial reports).
The paper's ablation (§4.3) shows it outperforms general BERT, RoBERTa, and Longformer on financial NER.

Given a token sequence $X = \{x_1, \ldots, x_n\}$:
$$H = \text{FinBERT}(X) \in \mathbb{R}^{n \times d}, \quad d=768$$

In [5]:
print(f"Loading FinBERT: {CFG['plm_name']}")
tokenizer = AutoTokenizer.from_pretrained(CFG["plm_name"])
plm_model  = AutoModel.from_pretrained(CFG["plm_name"]).to(DEVICE)
plm_model.eval()
PLM_DIM = plm_model.config.hidden_size
print(f"Hidden dimension: {PLM_DIM}")

nlp = spacy.load(CFG["spacy_model"])   # used for dep-parse in SSHG


@torch.no_grad()
def encode_sentences(texts: list[str]) -> np.ndarray:
    """CLS-pooled, L2-normalised sentence embeddings. Shape: (n, PLM_DIM)."""
    vecs = []
    for i in range(0, len(texts), CFG["batch_size"]):
        batch = texts[i : i + CFG["batch_size"]]
        enc   = tokenizer(batch, return_tensors="pt", truncation=True,
                          max_length=CFG["max_seq_len"], padding=True)
        enc   = {k: v.to(DEVICE) for k, v in enc.items()}
        cls   = plm_model(**enc).last_hidden_state[:, 0, :].cpu().numpy()
        vecs.append(cls)
    mat = np.vstack(vecs)
    return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9)


@torch.no_grad()
def encode_entity(text: str) -> np.ndarray:
    """Single-entity CLS vector, L2-normalised."""
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=32)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    vec = plm_model(**enc).last_hidden_state[0, 0, :].cpu().numpy()
    return vec / (np.linalg.norm(vec) + 1e-9)


print("FinBERT encoder ready.")

Loading FinBERT: ProsusAI/finbert


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 712.13it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hidden dimension: 768
FinBERT encoder ready.


## 6. SSHG — Syntax-Semantics Hybrid Graph (Paper §3.1)

Combines two graph signals per article:
- **Syntactic $G_{syn}$**: dependency arcs from spaCy, weight $\lambda_1=1.0$
- **Semantic $G_{sem}$**: sliding co-occurrence window, weight $\lambda_2 \cdot \cos(h_i, h_j)$

$G_{SSHG} = G_{syn} \cup G_{sem}$ — the hybrid is passed to the GNN.

In [6]:
def build_sshg(doc, word_embeddings: np.ndarray) -> nx.DiGraph:
    G = nx.DiGraph()
    n = len(doc)
    entity_token_map = {idx: ent.label_
                        for ent in doc.ents if ent.label_ in FINANCIAL_ENT_TYPES
                        for idx in range(ent.start, ent.end)}

    for i, tok in enumerate(doc):
        G.add_node(i, text=tok.text, lemma=tok.lemma_, pos=tok.pos_,
                   is_entity=(i in entity_token_map),
                   label=entity_token_map.get(i, ""))

    # Syntactic edges
    for tok in doc:
        if tok.i < n and tok.head.i < n:
            G.add_edge(tok.head.i, tok.i,
                       weight=CFG["lambda_syn"], edge_type="syn", dep=tok.dep_)

    # Semantic edges
    emb_n, W = min(n, len(word_embeddings)), CFG["window_size"]
    for i in range(emb_n):
        for j in range(i + 1, min(i + W + 1, emb_n)):
            if not G.has_edge(i, j):
                norm = (np.linalg.norm(word_embeddings[i]) *
                        np.linalg.norm(word_embeddings[j]))
                cos  = float(np.dot(word_embeddings[i], word_embeddings[j]) / norm) if norm > 1e-9 else 0.0
                w    = CFG["lambda_sem"] * cos
                if w > 0:
                    G.add_edge(i, j, weight=w, edge_type="sem")
                    G.add_edge(j, i, weight=w, edge_type="sem")
    return G


def graph_to_adj(G: nx.DiGraph, n: int) -> torch.Tensor:
    adj = np.zeros((n, n), dtype=np.float32)
    for u, v, d in G.edges(data=True):
        if u < n and v < n:
            adj[u, v] = d.get("weight", 1.0)
    np.fill_diagonal(adj, 1.0)
    s = adj.sum(axis=1, keepdims=True); s[s == 0] = 1.0
    return torch.tensor(adj / s, dtype=torch.float32)


print("SSHG builder ready.")

SSHG builder ready.


## 7. GNN Propagation (Paper §3.1)

$$h_i^{(l+1)} = \sigma\!\left(\sum_{j \in \mathcal{N}(i)} e_w(e_{ij}) \cdot W^{(l)} h_j^{(l)}\right)$$

Two layers ($L=2$). Output: $H_{GNN} \in \mathbb{R}^{n \times h}$.

In [7]:
class GNNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.W    = nn.Linear(in_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, H: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        return F.relu(self.norm(self.W(torch.mm(adj, H))))


class GNN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_layers: int = 2):
        super().__init__()
        dims = [in_dim] + [hidden_dim] * n_layers
        self.layers = nn.ModuleList(
            [GNNLayer(dims[i], dims[i+1]) for i in range(n_layers)])
    def forward(self, H: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        for layer in self.layers: H = layer(H, adj)
        return H


gnn = GNN(PLM_DIM, CFG["gnn_hidden"], CFG["gnn_layers"]).to(DEVICE)
print(f"GNN: {sum(p.numel() for p in gnn.parameters()):,} parameters")

GNN: 263,168 parameters


## 8. EARE — Entity-Aware Representation Enhancement (Paper §3.2)

Three sub-steps before GNN propagation:
1. **Entity pooling**: $\tilde{e}_i = \text{mean}(h_{s_i}\ldots h_{t_i})$
2. **Entity attention**: each token attends to all entity spans
3. **Entity gating**: $\hat{h}_j = g_j \odot (h_j + context_j) + (1-g_j)\odot h_j$

In [8]:
class EAREModule(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.W_a = nn.Linear(dim, dim, bias=False)
        self.W_g = nn.Linear(dim * 2, dim, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, H: torch.Tensor, entity_spans: list) -> torch.Tensor:
        n, d = H.shape
        if not entity_spans: return H
        entity_reps = [H[max(0,s):min(n,e)].mean(0, keepdim=True)
                       for s, e in entity_spans if max(0,s) < min(n,e)]
        if not entity_reps: return H
        E       = torch.cat(entity_reps, 0)                      # (m, d)
        context = torch.mm(F.softmax(torch.mm(self.W_a(H), E.T), dim=1), E)
        H_hat   = H + context
        gate    = torch.sigmoid(self.W_g(torch.cat([H_hat, E.mean(0).expand(n,-1)], 1)))
        return self.norm(gate * H_hat + (1 - gate) * H)


eare = EAREModule(PLM_DIM).to(DEVICE)
print(f"EARE: {sum(p.numel() for p in eare.parameters()):,} parameters")

EARE: 1,771,008 parameters


## 9. BIO Entity Classifier

MLP head over GNN-enriched token representations → BIO tag sequence → decoded entity spans.

In [9]:
ENT_TYPES_LIST = sorted(FINANCIAL_ENT_TYPES)
BIO_LABELS     = ["O"] + [f"B-{t}" for t in ENT_TYPES_LIST] + [f"I-{t}" for t in ENT_TYPES_LIST]
LABEL2ID       = {l: i for i, l in enumerate(BIO_LABELS)}
ID2LABEL       = {i: l for l, i in LABEL2ID.items()}
N_LABELS       = len(BIO_LABELS)


class EntityClassificationHead(nn.Module):
    def __init__(self, gnn_dim: int, n_labels: int):
        super().__init__()
        self.clf = nn.Sequential(
            nn.Linear(gnn_dim, gnn_dim // 2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(gnn_dim // 2, n_labels))
    def forward(self, H): return self.clf(H)


entity_head = EntityClassificationHead(CFG["gnn_hidden"], N_LABELS).to(DEVICE)


def decode_bio_tags(tokens: list, bio_ids: list) -> list[dict]:
    entities, cur, cur_type, cur_start = [], None, None, None
    for i, tag_id in enumerate(bio_ids):
        tag = ID2LABEL[tag_id]
        if tag.startswith("B-"):
            if cur: entities.append({"text": " ".join(cur), "label": cur_type,
                                      "start_token": cur_start, "end_token": i})
            cur, cur_type, cur_start = [tokens[i]], tag[2:], i
        elif tag.startswith("I-") and cur and tag[2:] == cur_type:
            cur.append(tokens[i])
        else:
            if cur: entities.append({"text": " ".join(cur), "label": cur_type,
                                      "start_token": cur_start, "end_token": i})
            cur, cur_type, cur_start = None, None, None
    if cur: entities.append({"text": " ".join(cur), "label": cur_type,
                              "start_token": cur_start, "end_token": len(bio_ids)})
    return entities


print(f"BIO classifier ready ({N_LABELS} labels).")

BIO classifier ready (25 labels).


## 10. Entity Extraction — Full Integration

`extract_entities(text)` runs the complete FinBERT → EARE → SSHG → GNN → BIO pipeline
and returns `Entity` dataclass objects compatible with the NEL stage.

GNN predictions are merged with spaCy NER spans to maximise recall on financial entity types.

In [10]:
@torch.no_grad()
def extract_entities(text: str) -> List[Entity]:
    """
    FinBERT → EARE → SSHG → GNN → BIO → Entity list.
    Returns List[Entity] ready for NEL linking.
    """
    doc    = nlp(text[:1500])
    tokens = [tok.text for tok in doc]
    n      = len(tokens)
    if n == 0: return []

    # spaCy NER baseline (for EARE entity spans + merge)
    spacy_ents = [
        Entity(text=ent.text, type=ent.label_, span=(ent.start_char, ent.end_char))
        for ent in doc.ents if ent.label_ in FINANCIAL_ENT_TYPES
    ]
    entity_spans = [(e.start, e.end) for e in doc.ents if e.label_ in FINANCIAL_ENT_TYPES]

    # FinBERT encoding
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=CFG["max_seq_len"], padding=False,
                    return_offsets_mapping=True)
    enc_in = {k: v.to(DEVICE) for k, v in enc.items() if k != "offset_mapping"}
    H_sub  = plm_model(**enc_in).last_hidden_state[0]   # (subword_len, d)

    # Subword → word alignment (first-subword rule)
    offsets   = enc["offset_mapping"][0].tolist()
    w2s       = [si for si, (s, e) in enumerate(offsets) if not (s == 0 and e == 0)][:n]
    n_aligned = len(w2s)
    if n_aligned == 0: return spacy_ents

    H_word = H_sub[w2s]   # (n_aligned, PLM_DIM)

    # EARE
    valid_spans = [(s, e) for s, e in entity_spans if s < n_aligned and e <= n_aligned]
    H_eare      = eare(H_word, valid_spans)

    # SSHG + GNN
    G   = build_sshg(doc, H_eare.cpu().numpy())
    adj = graph_to_adj(G, n_aligned).to(DEVICE)
    H_g = gnn(H_eare, adj)

    # BIO classification
    bio_ids    = entity_head(H_g).argmax(dim=1).cpu().tolist()
    gnn_spans  = decode_bio_tags(tokens[:n_aligned], bio_ids)
    gnn_ents   = [Entity(text=e["text"], type=e["label"],
                         span=(e["start_token"], e["end_token"]))
                  for e in gnn_spans if e.get("label") in FINANCIAL_ENT_TYPES]

    # Merge: prefer GNN entities; fall back to spaCy for any not already covered
    seen, merged = set(), []
    for ent in gnn_ents + spacy_ents:
        key = (ent.text.lower(), ent.type)
        if key not in seen:
            seen.add(key); merged.append(ent)
    return merged


# Quick sanity check
_sample = df[TEXT_COL].iloc[0]
_ents   = extract_entities(_sample)
print(f"Sample extraction: {len(_ents)} entities")
for e in _ents[:8]:
    print(f"  [{e.type:10s}]  {e.text}")

Sample extraction: 127 entities
  [NORP      ]  Marriott
  [NORP      ]  (
  [NORP      ]  MAR
  [NORP      ]  )
  [NORP      ]  ,
  [NORP      ]  the
  [NORP      ]  largest
  [NORP      ]  publicly


## 11. NEL Module (Hierarchical Linking)

In [11]:
from rapidfuzz import fuzz, process as rf_process

ALIAS_TABLE: Dict[str, str] = {
    'Goldman Sachs': 'LEI:549300IUPX6KIT9GK103', 'Goldman': 'LEI:549300IUPX6KIT9GK103',
    'GS': 'LEI:549300IUPX6KIT9GK103', 'Apple': 'GID:AAPL', 'Apple Inc': 'GID:AAPL',
    'Microsoft': 'GID:MSFT', 'JP Morgan': 'LEI:7H6GLXDRUGQFU57RNE97',
    'JPMorgan': 'LEI:7H6GLXDRUGQFU57RNE97', 'JP Morgan Chase': 'LEI:7H6GLXDRUGQFU57RNE97',
    'OpenAI': 'GID:OPENAI', 'First Republic Bank': 'GID:FRB',
    'Marcus': 'LEI:549300IUPX6KIT9GK103_MARCUS',
}

CANONICAL_DB: Dict[str, dict] = {
    'LEI:549300IUPX6KIT9GK103': {'canonical_id': 'LEI:549300IUPX6KIT9GK103',
        'name': 'Goldman Sachs Group Inc', 'type': 'parent', 'parent_id': None,
        'valid_time': {'start': '1869-01-01', 'end': None}},
    'LEI:549300IUPX6KIT9GK103_MARCUS': {'canonical_id': 'LEI:549300IUPX6KIT9GK103_MARCUS',
        'name': 'Marcus by Goldman Sachs', 'type': 'subsidiary',
        'parent_id': 'LEI:549300IUPX6KIT9GK103', 'valid_time': {'start': '2016-10-01', 'end': None}},
    'GID:AAPL': {'canonical_id': 'GID:AAPL', 'name': 'Apple Inc', 'type': 'parent',
        'parent_id': None, 'valid_time': {'start': '1977-01-01', 'end': None}},
    'GID:MSFT': {'canonical_id': 'GID:MSFT', 'name': 'Microsoft Corporation', 'type': 'parent',
        'parent_id': None, 'valid_time': {'start': '1975-04-04', 'end': None}},
    'LEI:7H6GLXDRUGQFU57RNE97': {'canonical_id': 'LEI:7H6GLXDRUGQFU57RNE97',
        'name': 'JPMorgan Chase & Co', 'type': 'parent', 'parent_id': None,
        'valid_time': {'start': '1799-01-01', 'end': None}},
}


class NEL:
    """Named Entity Linking: Entity text -> canonical_id (LEI / GID / LOCAL).
    Stages: ticker → alias → fuzzy → embedding stub → local fingerprint.
    """
    def __init__(self, ticker_anchors: Dict[str, str], strategy: str = 'fuzzy_flat',
                 fuzzy_threshold: float = 75.0):
        self.ticker_anchors = dict(ticker_anchors)
        self.strategy = strategy
        self.fuzzy_threshold = fuzzy_threshold
        self._alias_keys = list(ALIAS_TABLE)

    def link(self, entity: Entity, context: str = '') -> Link:
        canonical_id: Optional[str] = None
        confidence = 0.0

        cid = self.ticker_anchors.get(entity.text.upper())
        if cid: canonical_id, confidence = cid, 1.0

        if not canonical_id:
            cid = ALIAS_TABLE.get(entity.text)
            if cid: canonical_id, confidence = cid, 0.95

        if not canonical_id:
            match, score, _ = rf_process.extractOne(entity.text, self._alias_keys, scorer=fuzz.token_sort_ratio)
            if score >= self.fuzzy_threshold:
                canonical_id, confidence = ALIAS_TABLE[match], score / 100.0

        if not canonical_id:
            canonical_id = f"LOCAL:{hashlib.md5(entity.text.lower().encode()).hexdigest()[:8]}"
            confidence = 0.0

        valid_time = CANONICAL_DB.get(canonical_id, {}).get('valid_time', {'start': None, 'end': None})
        return Link(entity=entity.text, canonical_id=canonical_id,
                    confidence=confidence, valid_time=valid_time)

    def link_all(self, entities: List[Entity], context: str = '') -> List[Link]:
        return [self.link(e, context) for e in entities]


_nel_test = NEL(ticker_anchors={'GS': 'LEI:549300IUPX6KIT9GK103'})
_lnk = _nel_test.link(Entity('Goldman Sachs', 'ORG', (0, 13)))
print(f'Link: {_lnk.entity} -> {_lnk.canonical_id} ({_lnk.confidence:.2f})')


Link: Goldman Sachs -> LEI:549300IUPX6KIT9GK103 (0.95)


## 12. . Temporal Layer (Point-in-Time Enforcement)

In [12]:
from dateutil import parser as dateparser

_DATE_PATS = [
    re.compile(r'\b(\d{4}-\d{2}-\d{2})\b'),
    re.compile(r'\b(\w+ \d{1,2},?\s+\d{4})\b'),
    re.compile(r'\b(Q[1-4]\s+\d{4})\b', re.IGNORECASE),
    re.compile(r'\b(early|mid|late)\s+(\d{4})\b', re.IGNORECASE),
    re.compile(r'\b(\d{4})\b'),
]


class TemporalLayer:
    def extract_dates(self, text: str) -> List[Tuple[str, int, int]]:
        found = []
        for pat in _DATE_PATS:
            for m in pat.finditer(text):
                found.append((m.group(), m.start(), m.end()))
        found.sort(key=lambda x: x[1])
        deduped, last_end = [], -1
        for item in found:
            if item[1] >= last_end:
                deduped.append(item)
                last_end = item[2]
        return deduped

    def normalize_date(self, date_str: str) -> Optional[str]:
        s = date_str.strip()
        if re.fullmatch(r'\d{4}-\d{2}-\d{2}', s):
            return s

        # Quarter format: Q1 2022 -> 2022-01-01
        q = re.fullmatch(r'Q([1-4])\s+(\d{4})', s, re.IGNORECASE)
        if q:
            year = q.group(2)
            month = (int(q.group(1)) - 1) * 3 + 1
            return f'{year}-{month:02d}-01'

        # Coarse period: early/mid/late 2023
        p = re.fullmatch(r'(early|mid|late)\s+(\d{4})', s, re.IGNORECASE)
        if p:
            month_map = {'early': '01', 'mid': '06', 'late': '10'}
            return f"{p.group(2)}-{month_map[p.group(1).lower()]}-01"

        # Year-only: 2024 -> 2024-01-01
        if re.fullmatch(r'\d{4}', s):
            return f'{s}-01-01'

        # Fallback parser 
        try:
            dt = dateparser.parse(s, fuzzy=True)
            return dt.strftime('%Y-%m-%d') if dt else None
        except Exception:
            return None

    def infer_relation_time(self, sentence: str, doc_date: Optional[str] = None) -> Dict[str, Optional[str]]:
        normalized = sorted(filter(None, (self.normalize_date(d[0]) for d in self.extract_dates(sentence))))
        if normalized:
            return {'start': normalized[0], 'end': normalized[-1] if len(normalized) > 1 else None}
        return {'start': doc_date, 'end': None}

    def is_valid(self, relation_time: Dict, entity_valid_time: Dict) -> bool:
        r_start = relation_time.get('start')
        e_start, e_end = entity_valid_time.get('start'), entity_valid_time.get('end')
        if not r_start: return True
        if e_end and r_start > e_end: return False
        if e_start and r_start < e_start: return False
        return True


_tl = TemporalLayer()
print(_tl.normalize_date('Q1 2022'), _tl.normalize_date('early 2023'))

2022-01-01 2023-01-01


## 13. Relation Extraction — Bilinear Head + Pattern Prior (Primary)

**Primary relation extractor**. Three-stage scoring:
1. **Type-pair filter** — mask relations incompatible with entity types
2. **Pattern-rule prior** — lexical trigger in ±300-char context → confidence 0.85 (override)
3. **Neural blend** — $s = \alpha \cdot s_{proto} + (1-\alpha) \cdot s_{bili}$, threshold $\theta=0.30$

where $s_{proto}$ = cosine similarity to FinBERT relation prototype embeddings and
$s_{bili}$ = $\text{MLP}([h_{subj}; h_{obj}; h_{subj}\odot h_{obj}])$.

In [13]:
# ── Relation taxonomy ─────────────────────────────────────────────────────────
RELATIONS = {
    "ACQUIRED":          {"desc": "Company A acquired or bought company B.",
                          "valid": {("ORG","ORG")},
                          "pats": [r"acqui", r"takeover", r"bought", r"merger"]},
    "INVESTED_IN":       {"desc": "An org or person invested in or funded another company.",
                          "valid": {("ORG","ORG"),("PERSON","ORG")},
                          "pats": [r"invest", r"fund", r"financ", r"stake"]},
    "HOLDS_POSITION":    {"desc": "A person holds an executive role at a company.",
                          "valid": {("PERSON","ORG")},
                          "pats": [r"ceo",r"chief",r"president",r"director",r"officer",
                                   r"chairman",r"head of",r"founder",r"said"]},
    "LOCATED_IN":        {"desc": "An org is headquartered or located in a place.",
                          "valid": {("ORG","GPE"),("PERSON","GPE"),("FAC","GPE")},
                          "pats": [r"based in",r"headquarter",r"located"]},
    "REPORTED_REVENUE":  {"desc": "A company reported revenue, earnings, or financial results.",
                          "valid": {("ORG","MONEY"),("ORG","PERCENT"),("ORG","CARDINAL")},
                          "pats": [r"report",r"earn",r"revenue",r"profit",r"sales",r"forecast"]},
    "PARTNERED_WITH":    {"desc": "Two orgs formed a partnership or joint venture.",
                          "valid": {("ORG","ORG")},
                          "pats": [r"partner",r"alliance",r"joint venture",r"collaborat"]},
    "COMPETES_WITH":     {"desc": "Two companies are competitors in the same market.",
                          "valid": {("ORG","ORG")},
                          "pats": [r"rival",r"compet",r"versus",r"beat"]},
    "AFFECTS":           {"desc": "An event or policy affects an organization.",
                          "valid": {("EVENT","ORG"),("NORP","ORG"),("GPE","ORG"),("ORG","ORG")},
                          "pats": [r"affect",r"impact",r"hurt",r"boost",r"drag"]},
    "NO_RELATION":       {"desc": "Co-mentioned but no meaningful direct relation.",
                          "valid": None, "pats": []},
}
RELATION_NAMES = list(RELATIONS.keys())

# ── Prototype embeddings ───────────────────────────────────────────────────────
proto_norm = encode_sentences([RELATIONS[r]["desc"] for r in RELATION_NAMES])
print(f"Prototype matrix: {proto_norm.shape}  ({len(RELATION_NAMES)} relations)")

# ── Bilinear relation head ─────────────────────────────────────────────────────
class BilinearRelationHead(nn.Module):
    """[h_subj ; h_obj ; h_subj⊙h_obj] → n_relations logits."""
    def __init__(self, dim: int, n_rel: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim*3, dim*3//2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(dim*3//2, n_rel))
    def forward(self, hs, ho):
        return self.mlp(torch.cat([hs, ho, hs*ho], dim=1))

rel_head = BilinearRelationHead(PLM_DIM, len(RELATION_NAMES)).to(DEVICE)
print(f"BilinearRelationHead: {sum(p.numel() for p in rel_head.parameters()):,} parameters")


# ── Pattern-rule prior ────────────────────────────────────────────────────────
def pattern_match(text: str, subj: str, obj: str,
                  subj_type: str, obj_type: str) -> tuple[str, float]:
    lo   = text.lower()
    pos  = lo.find(subj.lower())
    ctx  = lo[max(0,pos-150):pos+300] if pos >= 0 else lo[:500]
    tpair = (subj_type, obj_type)
    for rel, meta in RELATIONS.items():
        if rel == "NO_RELATION": continue
        if meta["valid"] is not None and tpair not in meta["valid"]: continue
        for pat in meta["pats"]:
            if re.search(pat, ctx, re.IGNORECASE):
                return rel, 0.85
    return None, 0.0


# ── Full scoring pipeline ─────────────────────────────────────────────────────
def _softmax(x):
    x = x - np.max(x[np.isfinite(x)])
    ex = np.where(np.isfinite(x), np.exp(x), 0.0)
    return ex / (ex.sum() + 1e-9)


@torch.no_grad()
def classify_relation(subj: str, obj: str, subj_type: str, obj_type: str,
                       context: str) -> tuple[str, float]:
    """Returns (relation_name, confidence). Primary RE method."""
    tpair = (subj_type, obj_type)
    mask  = [True if r=="NO_RELATION" or RELATIONS[r]["valid"] is None
             else tpair in RELATIONS[r]["valid"] for r in RELATION_NAMES]
    if sum(mask[:-1]) == 0: return "NO_RELATION", 1.0

    # Pattern override
    pr, pc = pattern_match(context, subj, obj, subj_type, obj_type)
    if pr: return pr, pc

    # Encode
    hs = torch.tensor(encode_entity(subj)).unsqueeze(0).to(DEVICE)
    ho = torch.tensor(encode_entity(obj )).unsqueeze(0).to(DEVICE)

    # Proto similarity
    pair_vec = (hs.cpu().numpy()[0] + ho.cpu().numpy()[0]) / 2
    ms_proto = np.where(mask, proto_norm @ pair_vec, -np.inf)

    # Bilinear
    ms_bili = np.where(mask, rel_head(hs, ho)[0].cpu().numpy(), -np.inf)

    probs = CFG["alpha_blend"] * _softmax(ms_proto) + (1-CFG["alpha_blend"]) * _softmax(ms_bili)
    best_i, best_rel, best_conf = int(np.argmax(probs)), RELATION_NAMES[int(np.argmax(probs))], float(np.max(probs))

    if best_conf < CFG["relation_threshold"] or best_rel == "NO_RELATION":
        return "NO_RELATION", best_conf
    return best_rel, best_conf


# Sanity check
_r, _c = classify_relation("Marriott", "Arne Sorenson", "ORG", "PERSON",
                            "Marriott CEO Arne Sorenson said profits rose.")
print(f"Test → {_r} (conf={_c:.3f})")

Prototype matrix: (9, 768)  (9 relations)
BilinearRelationHead: 2,665,737 parameters
Test → NO_RELATION (conf=1.000)


## 14. Relation Extraction — Ollama LLM Joint Extractor (Supplementary)

Used for entity pairs where the bilinear head returns low confidence
or to discover relation types not in the taxonomy (open-domain extraction).
Requires `ollama run mistral` running locally.

`Ollama_RE` uses strict JSON schema, temperature=0, deterministic output.

In [14]:
# ── C. Ollama_RE (Llama 3.2 — joint LLM extraction) ──────────────────────────
class REBase(ABC):
    """Base interface for relation extractors."""
    @abstractmethod
    def extract(self, text: str, entities: List["Entity"]) -> List["Relation"]:
        raise NotImplementedError


class Ollama_RE(REBase):
    """Llama 3.2 via Ollama REST API. temperature=0, strict JSON contract.
    dev_mode=True -> llama3.2:1b for fast iteration.
    """

    RELATION_TYPES = [
        'acquired', 'merged_with', 'invested_in', 'partner_of', 'subsidiary_of',
        'sued', 'sold_stake_to', 'raised_funding_from', 'ceo_of', 'competes_with',
        'regulates', 'board_member_of',
    ]

    _PROMPT = """Extract ONLY valid financial relations as a JSON array.
Each element: {{"subj": str, "rel": str, "obj": str, "time": str|null, "provenance": str}}

Rules:
- resolve coreference before extraction
- no hallucinated entities
- provenance = exact source sentence
- rel MUST be one of: {rel_types}
- return [] if no valid relations found

Text: {text}
Known entities: {entities}

Output ONLY the JSON array, nothing else:"""

    def __init__(self, model: str = 'llama3.2:3b', dev_mode: bool = False,
                 ollama_url: str = 'http://localhost:11434'):
        self.model = 'llama3.2:1b' if dev_mode else model
        self.ollama_url = ollama_url

    def _call(self, prompt: str) -> str:
        resp = requests.post(f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0, 'top_p': 1}},
            timeout=120)
        resp.raise_for_status()
        return resp.json()['response']

    def extract(self, text: str, entities: List["Entity"]) -> List["Relation"]:
        prompt = self._PROMPT.format(
            rel_types=', '.join(self.RELATION_TYPES),
            text=text[:1500],
            entities=json.dumps([{'text': e.text, 'type': e.type} for e in entities])
        )
        try:
            raw = self._call(prompt)
            arr_match = re.search(r'\[.*\]', raw, re.DOTALL)
            if not arr_match:
                return []
            triples = json.loads(arr_match.group())
            rels = []
            for t in triples:
                if not all(k in t for k in ('subj', 'rel', 'obj')):
                    continue
                if t['rel'] not in self.RELATION_TYPES:
                    continue
                t_time = t.get('time')
                rels.append(Relation(
                    subj=t['subj'], rel=t['rel'], obj=t['obj'],
                    time={'start': t_time, 'end': None} if isinstance(t_time, str)
                         else (t_time or {'start': None, 'end': None}),
                    provenance=Provenance('', t.get('provenance', text[:300])),
                    confidence=float(t.get('confidence', 0.8))
                ))
            return rels
        except Exception as exc:
            print(f'[Ollama_RE] {exc}')
            return []


RE_REGISTRY: Dict[str, type] = {'ollama': Ollama_RE}
if 'OpenIE_RE' in globals():
    RE_REGISTRY['openie'] = OpenIE_RE
if 'Snowball_RE' in globals():
    RE_REGISTRY['snowball'] = Snowball_RE


def get_re(name: str, **kwargs) -> REBase:
    if name not in RE_REGISTRY:
        raise ValueError(f'Unknown RE: {name}. Available: {list(RE_REGISTRY)}')
    return RE_REGISTRY[name](**kwargs)

print('RE registry:', list(RE_REGISTRY))


def get_supplementary_re(ollama_url: str = None, dev_mode: bool = False):
    """Return an Ollama_RE extractor. Falls back gracefully if Ollama is unavailable."""
    url = ollama_url or CFG["ollama_url"]
    model = 'mistral' if not dev_mode else 'llama3.2:1b'
    return Ollama_RE(model=model, ollama_url=url)


RE registry: ['ollama']


## 15. . Provenance Enforcement

In [15]:
class ProvenanceEnforcer:
    @staticmethod
    def enforce(r: Relation, chunk_id: str, fallback: str = '') -> Relation:
        if not r.provenance.source_chunk_id: r.provenance.source_chunk_id = chunk_id
        if not r.provenance.raw_sentence: r.provenance.raw_sentence = fallback
        return r

    @staticmethod
    def validate(r: Relation) -> Tuple[bool, str]:
        if not r.provenance.source_chunk_id: return False, 'missing source_chunk_id'
        if not r.provenance.raw_sentence: return False, 'missing raw_sentence'
        return True, 'ok'

    @staticmethod
    def filter_valid(relations: List[Relation]) -> List[Relation]:
        out = []
        for r in relations:
            ok, reason = ProvenanceEnforcer.validate(r)
            if ok: out.append(r)
            else: print(f'[Provenance] Dropped ({r.subj},{r.rel},{r.obj}): {reason}')
        return out

print('ProvenanceEnforcer loaded.')


ProvenanceEnforcer loaded.


## 16. . Graph Builder (Versioned, Append-Only)

In [16]:
import networkx as nx
from collections import defaultdict


class GraphBuilder:
    """Append-only versioned knowledge graph.
    backend: 'networkx' (dev) | 'neo4j' (prod stub)
    scope:   'global' | 'document_local'
    Edge key: (subj_id, obj_id, rel, time_start) — parallel edges, never overwritten.
    """

    def __init__(self, backend: str = 'networkx', scope: str = 'global'):
        self.backend = backend
        self.scope = scope
        self._local: Dict[str, nx.MultiDiGraph] = {}
        self._global: nx.MultiDiGraph = nx.MultiDiGraph()
        self._edge_index: Dict[Tuple, List[dict]] = defaultdict(list)

    def _graph(self, doc_id: str) -> nx.MultiDiGraph:
        if self.scope == 'document_local':
            if doc_id not in self._local:
                self._local[doc_id] = nx.MultiDiGraph()
            return self._local[doc_id]
        return self._global

    def _add_node(self, G, cid, **attrs):
        if not G.has_node(cid): G.add_node(cid, **attrs)
        else: G.nodes[cid].update({k: v for k, v in attrs.items() if v is not None})

    def add_relation(self, doc_id: str, ls: Link, lo: Link, r: Relation) -> None:
        G = self._graph(doc_id)
        self._add_node(G, ls.canonical_id, name=ls.entity, valid_time=ls.valid_time, confidence=ls.confidence)
        self._add_node(G, lo.canonical_id, name=lo.entity, valid_time=lo.valid_time, confidence=lo.confidence)
        time_start = r.time.get('start') or 'unknown'
        idx_key = (ls.canonical_id, lo.canonical_id, r.rel, time_start)
        edge_data = {'rel': r.rel, 'time': r.time,
            'provenance': {'source_chunk_id': r.provenance.source_chunk_id,
                           'raw_sentence': r.provenance.raw_sentence},
            'confidence': r.confidence, 'doc_id': doc_id}
        edge_key = f"{r.rel}__{time_start}__{len(self._edge_index[idx_key])}"
        G.add_edge(ls.canonical_id, lo.canonical_id, key=edge_key, **edge_data)
        self._edge_index[idx_key].append(edge_data)

    def get_graph(self, doc_id: Optional[str] = None) -> nx.MultiDiGraph:
        if self.scope == 'document_local':
            if doc_id: return self._local.get(doc_id, nx.MultiDiGraph())
            merged = nx.MultiDiGraph()
            for g in self._local.values(): merged.update(g)
            return merged
        return self._global

    def stats(self) -> dict:
        G = self.get_graph()
        total = G.number_of_edges()
        with_prov = sum(1 for _,_,d in G.edges(data=True) if d.get('provenance',{}).get('source_chunk_id'))
        with_ts = sum(1 for _,_,d in G.edges(data=True) if d.get('time',{}).get('start'))
        return {'nodes': G.number_of_nodes(), 'edges': total,
                'provenance_pct': round(with_prov/max(total,1)*100,1),
                'timestamp_pct': round(with_ts/max(total,1)*100,1),
                'conflict_groups': len({k:v for k,v in self._edge_index.items() if len(v)>1})}

print('GraphBuilder loaded.')


GraphBuilder loaded.


## 17. . Community Detection (Leiden)

Leiden algorithm applied to the undirected projection of the knowledge graph.  
Falls back to `networkx.community.greedy_modularity_communities` if `leidenalg` is not installed.  
Produces a **3-level hierarchy**: leaf clusters (L0) → super-clusters (L1) → global (L2).


In [17]:
class CommunityDetector:
    """Detect communities with Leiden; build 3-level hierarchy of CommunityNode objects."""

    def __init__(self, resolution: float = 1.0, n_iterations: int = 10):
        self.resolution = resolution
        self.n_iterations = n_iterations

    # ── low-level: run Leiden (or networkx fallback) ──────────────────────────
    def _partition(self, G_und: nx.Graph) -> Dict[str, int]:
        nodes = list(G_und.nodes())
        try:
            import leidenalg, igraph as ig
            node_idx = {n: i for i, n in enumerate(nodes)}
            edges = [(node_idx[u], node_idx[v]) for u, v in G_und.edges()]
            ig_g = ig.Graph(n=len(nodes), edges=edges, directed=False)
            part = leidenalg.find_partition(
                ig_g, leidenalg.ModularityVertexPartition,
                n_iterations=self.n_iterations
            )
            return {nodes[i]: c for c, members in enumerate(part) for i in members}
        except ImportError:
            comms = list(nx.community.greedy_modularity_communities(G_und))
            return {node: i for i, comm in enumerate(comms) for node in comm}

    # ── public: map every node to its leaf cluster id ─────────────────────────
    def detect(self, G: nx.MultiDiGraph) -> Dict[str, int]:
        """Return {canonical_id -> cluster_id} for every node in G."""
        if G.number_of_nodes() == 0:
            return {}
        G_und = G.to_undirected()
        connected = [n for n in G_und.nodes() if G_und.degree(n) > 0]
        if not connected:
            return {n: 0 for n in G_und.nodes()}
        mapping = self._partition(G_und.subgraph(connected))
        # isolated nodes get cluster -1
        for n in G_und.nodes():
            mapping.setdefault(n, -1)
        return mapping

    # ── build 3-level hierarchy ───────────────────────────────────────────────
    def build_hierarchy(
        self,
        G: nx.MultiDiGraph,
        level0: Dict[str, int],
    ) -> Dict[int, CommunityNode]:
        communities: Dict[int, CommunityNode] = {}
        cluster_members: Dict[int, List[str]] = defaultdict(list)
        for node, cid in level0.items():
            cluster_members[cid].append(node)

        # Level 0 — leaf clusters
        for cid, members in cluster_members.items():
            key_rels = [
                {'subj': u, 'rel': d.get('rel',''), 'obj': v, 'time': d.get('time',{})}
                for u, v, d in G.edges(data=True)
                if u in members and v in members
            ][:20]
            communities[cid] = CommunityNode(
                cluster_id=f'L0_C{cid}', level=0, summary='',
                entities=list(members), key_relations=key_rels
            )

        # Level 1 — super-clusters: connected components of cluster adjacency
        cluster_adj: Dict[int, set] = defaultdict(set)
        for u, v in G.edges():
            cu, cv = level0.get(u, -1), level0.get(v, -1)
            if cu != cv and cu != -1 and cv != -1:
                cluster_adj[cu].add(cv)
                cluster_adj[cv].add(cu)

        visited: set = set()
        sc_id = 0
        node_to_sc: Dict[int, int] = {}
        for cid in cluster_members:
            if cid in visited: continue
            queue, group = [cid], []
            while queue:
                curr = queue.pop()
                if curr in visited: continue
                visited.add(curr); group.append(curr)
                queue.extend(cluster_adj[curr] - visited)
            for g in group:
                node_to_sc[g] = sc_id
            sc_id += 1

        for sid in range(sc_id):
            children = [c for c, s in node_to_sc.items() if s == sid]
            all_ents = [e for c in children for e in communities[c].entities]
            all_rels = [r for c in children for r in communities[c].key_relations][:30]
            new_id = max(communities, default=-1) + 1
            communities[new_id] = CommunityNode(
                cluster_id=f'L1_SC{sid}', level=1, summary='',
                entities=all_ents, key_relations=all_rels
            )

        print(f'Hierarchy built: {sum(1 for n in communities.values() if n.level==0)} L0 clusters, '
              f'{sum(1 for n in communities.values() if n.level==1)} L1 super-clusters.')
        return communities


print('CommunityDetector loaded.')


CommunityDetector loaded.


## 18. . Community Summaries (LLM-Generated)

Each cluster gets an LLM-generated summary via Ollama.  
**Constraint**: summaries are grounded *only* in graph edges — no external knowledge.


In [18]:
class CommunitySummarizer:
    """Generate CommunityNode summaries via Ollama. Grounded in graph edges only."""

    _PROMPT = """You are a financial analyst. Summarize this entity cluster using ONLY the relations listed.
Do NOT add external knowledge. Do NOT invent facts not present in the relations.

Cluster ID: {cluster_id}
Level: {level}
Entities: {entities}
Relations:
{relations}

Respond with ONLY this JSON:
{{
  "cluster_id": "{cluster_id}",
  "level": {level},
  "summary": "<2-3 sentences grounded strictly in the relations above>",
  "entities": {entities_json},
  "key_relations": {rels_json}
}}"""

    def __init__(self, model: str = 'llama3.2:3b', dev_mode: bool = False,
                 ollama_url: str = 'http://localhost:11434'):
        self.model = 'llama3.2:1b' if dev_mode else model
        self.ollama_url = ollama_url

    def _call(self, prompt: str) -> str:
        resp = requests.post(f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0}},
            timeout=120)
        resp.raise_for_status()
        return resp.json()['response']

    def summarize(self, node: CommunityNode) -> CommunityNode:
        rels_str = '\n'.join(
            f'  ({r["subj"]}) --[{r["rel"]}]--> ({r["obj"]})'
            + (f' @ {r["time"].get("start","?")}' if r.get('time') else '')
            for r in node.key_relations[:15]
        ) or '  (no intra-cluster relations)'

        prompt = self._PROMPT.format(
            cluster_id=node.cluster_id, level=node.level,
            entities=node.entities[:20], relations=rels_str,
            entities_json=json.dumps(node.entities[:20]),
            rels_json=json.dumps(node.key_relations[:10])
        )
        try:
            raw = self._call(prompt)
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                data = json.loads(m.group())
                node.summary = data.get('summary', '')
        except Exception as exc:
            print(f'[CommunitySummarizer] {node.cluster_id}: {exc}')
            node.summary = f'[Summarization failed for {node.cluster_id}]'
        return node

    def summarize_all(self, communities: Dict[int, CommunityNode]) -> Dict[int, CommunityNode]:
        print(f'Summarizing {len(communities)} community nodes...')
        for cid in sorted(communities, key=lambda k: communities[k].level):
            node = communities[cid]
            print(f'  {node.cluster_id} (L{node.level}, {len(node.entities)} entities)')
            communities[cid] = self.summarize(node)
        return communities


print('CommunitySummarizer loaded.')


CommunitySummarizer loaded.


## 19. 0. Dual Retrieval System

| Mode | Trigger | Mechanism |
|------|---------|-----------|
| **Local** (KG-RAG) | Query contains named entities | Graph traversal from seed nodes |
| **Global** (Hierarchical) | Thematic / no entities | Keyword-score cluster summaries |
| **Hybrid** | Explicit | Local context + Global context merged |

Routing is automatic (`mode='auto'`): entity presence → local, else → global.


In [19]:
class LocalRetriever:
    """KG-RAG: entity extraction → subgraph traversal → natural-language context."""

    def __init__(self, graph: nx.MultiDiGraph, nel: NEL,
                 max_hops: int = 2, max_nodes: int = 25):
        self.graph = graph
        self.nel = nel
        self.max_hops = max_hops
        self.max_nodes = max_nodes

    def _query_entities(self, query: str) -> List[str]:
        """Map query mentions to canonical_ids via ticker + alias lookup."""
        found = []
        for m in TICKER_PATTERN.finditer(query):
            found.append(f'TICKER:{m.group(1) or m.group(2)}')
        for alias, cid in ALIAS_TABLE.items():
            if alias.lower() in query.lower():
                found.append(cid)
        return list(set(found))

    def _traverse(self, seeds: List[str]) -> nx.MultiDiGraph:
        seeds = [s for s in seeds if s in self.graph]
        if not seeds:
            return nx.MultiDiGraph()
        visited = set(seeds)
        frontier = list(seeds)
        for _ in range(self.max_hops):
            nxt = []
            for node in frontier:
                for nbr in list(self.graph.successors(node)) + list(self.graph.predecessors(node)):
                    if nbr not in visited:
                        visited.add(nbr); nxt.append(nbr)
            frontier = nxt
            if len(visited) >= self.max_nodes: break
        return self.graph.subgraph(list(visited)[:self.max_nodes]).copy()

    def _to_context(self, sg: nx.MultiDiGraph) -> str:
        if sg.number_of_edges() == 0:
            return 'No graph context found for the query entities.'
        lines = ['[Local KG Context]']
        for u, v, d in sg.edges(data=True):
            un = sg.nodes[u].get('name', u)
            vn = sg.nodes[v].get('name', v)
            ts = d.get('time', {}).get('start', '?')
            prov = d.get('provenance', {}).get('raw_sentence', '')[:100]
            lines.append(f'  ({un}) --[{d.get("rel","?")}]--> ({vn}) @ {ts}')
            if prov: lines.append(f'    \""{prov}\"')
        return '\n'.join(lines)

    def retrieve(self, query: str) -> dict:
        eids = self._query_entities(query)
        sg = self._traverse(eids)
        return {'context': self._to_context(sg), 'retrieval_mode': 'local',
                'entity_ids': eids, 'subgraph_nodes': sg.number_of_nodes()}


class GlobalRetriever:
    """Hierarchical RAG: keyword-score cluster summaries → retrieve top-k → context."""

    def __init__(self, communities: Dict[int, CommunityNode], top_k: int = 3):
        self.communities = communities
        self.top_k = top_k

    def _score(self, query: str, node: CommunityNode) -> float:
        qterms = set(query.lower().split())
        s_score = len(qterms & set(node.summary.lower().split())) / (len(qterms) + 1)
        e_terms = {t.lower() for e in node.entities
                   for t in e.replace(':', ' ').replace('_', ' ').split()}
        e_score = len(qterms & e_terms) / (len(qterms) + 1)
        r_terms = {t.lower() for r in node.key_relations
                   for t in (r.get('subj','') + ' ' + r.get('obj','')).split()}
        r_score = len(qterms & r_terms) / (len(qterms) + 1)
        return s_score * 0.5 + e_score * 0.3 + r_score * 0.2

    def retrieve(self, query: str) -> dict:
        if not self.communities:
            return {'context': '[No communities available]', 'retrieval_mode': 'global', 'clusters': []}
        scored = sorted(
            [(cid, n, self._score(query, n)) for cid, n in self.communities.items()],
            key=lambda x: x[2], reverse=True
        )[:self.top_k]
        lines = ['[Global Hierarchical Context]']
        clusters_used = []
        for _, node, score in scored:
            if not node.summary: continue
            lines.append(f'\n[{node.cluster_id}] L{node.level} (score={score:.3f}):')
            lines.append(f'  {node.summary}')
            lines.append(f'  Entities: {", ".join(str(e) for e in node.entities[:8])}')
            clusters_used.append(node.cluster_id)
        return {'context': '\n'.join(lines), 'retrieval_mode': 'global', 'clusters': clusters_used}


class DualRetriever:
    """Routes to Local or Global retrieval; generates answer via Ollama.
    Routing: entity found in query → local; else → global.
    """

    def __init__(self, local: LocalRetriever, global_: GlobalRetriever,
                 model: str = 'llama3.2:3b', ollama_url: str = 'http://localhost:11434'):
        self.local = local
        self.global_ = global_
        self.model = model
        self.ollama_url = ollama_url

    def _has_entities(self, query: str) -> bool:
        if TICKER_PATTERN.search(query): return True
        return any(a.lower() in query.lower() for a in ALIAS_TABLE)

    def retrieve(self, query: str, mode: str = 'auto') -> dict:
        if mode == 'local': return self.local.retrieve(query)
        if mode == 'global': return self.global_.retrieve(query)
        if mode == 'hybrid':
            l, g = self.local.retrieve(query), self.global_.retrieve(query)
            return {'context': l['context'] + '\n\n' + g['context'],
                    'retrieval_mode': 'hybrid', 'local': l, 'global': g}
        # auto
        return self.local.retrieve(query) if self._has_entities(query) else self.global_.retrieve(query)

    def answer(self, query: str, mode: str = 'auto') -> dict:
        ret = self.retrieve(query, mode)
        prompt = (
            'Answer the question using ONLY the provided knowledge graph context.\n'
            'If context is insufficient, say "Insufficient context."\n\n'
            f'Context:\n{ret["context"]}\n\nQuestion: {query}\n\nAnswer:'
        )
        try:
            resp = requests.post(f'{self.ollama_url}/api/generate',
                json={'model': self.model, 'prompt': prompt, 'stream': False,
                      'options': {'temperature': 0}},
                timeout=60)
            resp.raise_for_status()
            answer = resp.json()['response']
        except Exception as exc:
            answer = f'[Answer generation failed: {exc}]'
        return {'query': query, 'answer': answer,
                'context': ret['context'], 'retrieval_mode': ret['retrieval_mode']}


print('LocalRetriever, GlobalRetriever, DualRetriever loaded.')


LocalRetriever, GlobalRetriever, DualRetriever loaded.


## 20. b. Advanced Retrieval — LLM-Guided + Semantic

Upgrades over the rule-based `LocalRetriever` / keyword `GlobalRetriever`:

| Class | Upgrade |
|-------|---------|
| **`LLM_LocalRetriever`** | `llama3.2:3b` extracts query entities; guided multi-hop prunes irrelevant neighbours |
| **`SemanticGlobalRetriever`** | `sentence-transformers` cosine similarity replaces keyword overlap; falls back gracefully |
| **`AdvancedDualRetriever`** | New `llm_hybrid` mode: LLM synthesises + re-ranks both local and global contexts |

In [20]:
# ── A. LLM_LocalRetriever ────────────────────────────────────────────────────
class LLM_LocalRetriever(LocalRetriever):
    """Extends LocalRetriever with llama3.2:3b-guided entity extraction.

    Two upgrades over rule-based LocalRetriever:
      1. LLM extracts query entities  (not just ticker regex / alias table)
      2. LLM prunes neighbour expansion (guided multi-hop vs blind BFS)
    """

    _ENTITY_PROMPT = (
        "Identify all named entities (organizations, people, financial instruments, products) "
        "in the following query. Return ONLY a JSON array of strings. No explanation.\n\n"
        "Query: {query}\n\n"
        "Known entity names for context (match if applicable):\n{known_ids}\n\n"
        "Output ONLY JSON array, e.g. [\"Goldman Sachs\", \"Apple\"]:"
    )

    _PRUNE_PROMPT = (
        "Given a knowledge graph traversal for the query below, "
        "select which neighbour nodes are most relevant to expand.\n\n"
        "Query: {query}\n"
        "Current nodes: {current_nodes}\n"
        "Candidate neighbours: {candidates}\n\n"
        "Return ONLY a JSON array of node names to expand (max {max_expand} items).\n"
        "Output ONLY the JSON array:"
    )

    def __init__(self, graph: nx.MultiDiGraph, nel: NEL,
                 max_hops: int = 2, max_nodes: int = 25,
                 model: str = 'llama3.2:3b',
                 ollama_url: str = 'http://localhost:11434'):
        super().__init__(graph, nel, max_hops, max_nodes)
        self.model = model
        self.ollama_url = ollama_url

    def _llm_call(self, prompt: str, timeout: int = 30) -> str:
        resp = requests.post(
            f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0, 'top_p': 1}},
            timeout=timeout
        )
        resp.raise_for_status()
        return resp.json()['response']

    def _llm_query_entities(self, query: str) -> List[str]:
        """Use llama3.2:3b to extract entity mentions, then resolve to canonical_ids."""
        known = list(ALIAS_TABLE.keys())[:20]
        prompt = self._ENTITY_PROMPT.format(query=query, known_ids=json.dumps(known))
        try:
            raw = self._llm_call(prompt)
            arr = re.search(r'\[.*?\]', raw, re.DOTALL)
            if not arr:
                return self._query_entities(query)   # rule-based fallback
            mentions = json.loads(arr.group())
            from rapidfuzz import process as rf_proc, fuzz as _fuzz
            resolved = []
            for mention in mentions:
                cid = ALIAS_TABLE.get(mention)
                if cid:
                    resolved.append(cid)
                    continue
                mr = rf_proc.extractOne(mention, list(ALIAS_TABLE.keys()),
                                        scorer=_fuzz.token_sort_ratio)
                if mr and mr[1] >= 70:
                    resolved.append(ALIAS_TABLE[mr[0]])
                else:
                    local_id = f"LOCAL:{hashlib.md5(mention.lower().encode()).hexdigest()[:8]}"
                    if local_id in self.graph:
                        resolved.append(local_id)
            return list(set(resolved + self._query_entities(query)))
        except Exception as exc:
            print(f'[LLM_LocalRetriever] entity extraction failed: {exc}, using rule-based')
            return self._query_entities(query)

    def _llm_guided_traverse(self, seeds: List[str], query: str) -> nx.MultiDiGraph:
        """Multi-hop traversal where llama3.2:3b prunes which neighbours to expand."""
        seeds = [s for s in seeds if s in self.graph]
        if not seeds:
            return nx.MultiDiGraph()

        visited = set(seeds)
        frontier = list(seeds)

        for hop in range(self.max_hops):
            candidates: Dict[str, str] = {}
            for node in frontier:
                for nbr in list(self.graph.successors(node)) + list(self.graph.predecessors(node)):
                    if nbr not in visited:
                        candidates[nbr] = self.graph.nodes[nbr].get('name', nbr)

            if not candidates:
                break

            max_expand = max(1, self.max_nodes - len(visited))
            if len(candidates) <= max_expand:
                next_frontier = list(candidates.keys())
            else:
                current_names = [self.graph.nodes[n].get('name', n) for n in visited]
                prompt = self._PRUNE_PROMPT.format(
                    query=query,
                    current_nodes=json.dumps(current_names[:15]),
                    candidates=json.dumps(list(candidates.values())[:20]),
                    max_expand=max_expand,
                )
                try:
                    raw = self._llm_call(prompt)
                    arr = re.search(r'\[.*?\]', raw, re.DOTALL)
                    selected_names = set(json.loads(arr.group())) if arr else set()
                    name_to_id = {v: k for k, v in candidates.items()}
                    next_frontier = [name_to_id[n] for n in selected_names if n in name_to_id]
                    if not next_frontier:
                        next_frontier = list(candidates.keys())[:max_expand]
                except Exception as exc:
                    print(f'[LLM_LocalRetriever] prune failed hop {hop}: {exc}')
                    next_frontier = list(candidates.keys())[:max_expand]

            visited.update(next_frontier)
            frontier = next_frontier
            if len(visited) >= self.max_nodes:
                break

        return self.graph.subgraph(list(visited)[:self.max_nodes]).copy()

    def retrieve(self, query: str) -> dict:
        eids = self._llm_query_entities(query)
        sg   = self._llm_guided_traverse(eids, query)
        return {
            'context':        self._to_context(sg),
            'retrieval_mode': 'llm_local',
            'entity_ids':     eids,
            'subgraph_nodes': sg.number_of_nodes(),
        }


# ── B. SemanticGlobalRetriever ────────────────────────────────────────────────
class SemanticGlobalRetriever(GlobalRetriever):
    """Cosine-similarity retrieval over community summaries.
    Requires sentence-transformers; falls back to keyword scoring if unavailable.
    """

    def __init__(self, communities: Dict[int, CommunityNode],
                 top_k: int = 3, embed_model: str = 'all-MiniLM-L6-v2'):
        super().__init__(communities, top_k)
        self._embed_model_name = embed_model
        self._encoder         = None
        self._cluster_embs    = None
        self._cluster_order: List[int] = []
        self._build_index()

    def _build_index(self):
        try:
            from sentence_transformers import SentenceTransformer
            self._encoder = SentenceTransformer(self._embed_model_name)
            summaries, self._cluster_order = [], []
            for cid, node in self.communities.items():
                text = node.summary or ' '.join(str(e) for e in node.entities[:10])
                summaries.append(text)
                self._cluster_order.append(cid)
            if summaries:
                self._cluster_embs = self._encoder.encode(summaries, convert_to_tensor=True)
            print(f'[SemanticGlobalRetriever] Indexed {len(summaries)} clusters.')
        except ImportError:
            print('[SemanticGlobalRetriever] sentence-transformers not installed — keyword fallback active.')

    def retrieve(self, query: str) -> dict:
        if self._encoder is None or self._cluster_embs is None:
            return super().retrieve(query)
        try:
            from sentence_transformers import util
            q_emb = self._encoder.encode(query, convert_to_tensor=True)
            cos   = util.cos_sim(q_emb, self._cluster_embs)[0]
            top_i = cos.argsort(descending=True)[:self.top_k].tolist()
            lines, clusters_used = ['[Global Semantic Context]'], []
            for idx in top_i:
                cid  = self._cluster_order[idx]
                node = self.communities[cid]
                if not node.summary:
                    continue
                score = float(cos[idx])
                lines.append(f'\n[{node.cluster_id}] L{node.level} (cos_sim={score:.3f}):')
                lines.append(f'  {node.summary}')
                lines.append(f'  Entities: {", ".join(str(e) for e in node.entities[:8])}')
                clusters_used.append(node.cluster_id)
            return {'context': '\n'.join(lines), 'retrieval_mode': 'semantic_global',
                    'clusters': clusters_used}
        except Exception as exc:
            print(f'[SemanticGlobalRetriever] {exc} — keyword fallback')
            return super().retrieve(query)


# ── C. AdvancedDualRetriever ──────────────────────────────────────────────────
class AdvancedDualRetriever(DualRetriever):
    """Drops in over DualRetriever.
    New mode 'llm_hybrid': llama3.2:3b merges + re-ranks local and global contexts.
    """

    _RERANK_PROMPT = (
        "You have two knowledge graph contexts for the query below.\n"
        "Synthesise them into a single concise context keeping only the most relevant facts.\n"
        "Do NOT add information not present in the contexts.\n\n"
        "Query: {query}\n\n"
        "Local Context:\n{local_ctx}\n\n"
        "Global Context:\n{global_ctx}\n\n"
        "Synthesised Context (max 400 words):"
    )

    def __init__(self, local: LLM_LocalRetriever,
                 global_: SemanticGlobalRetriever,
                 model: str = 'llama3.2:3b',
                 ollama_url: str = 'http://localhost:11434'):
        super().__init__(local, global_, model=model, ollama_url=ollama_url)
        self.local:   LLM_LocalRetriever    = local
        self.global_: SemanticGlobalRetriever = global_

    def _llm_call(self, prompt: str, timeout: int = 60) -> str:
        resp = requests.post(
            f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0}},
            timeout=timeout
        )
        resp.raise_for_status()
        return resp.json()['response']

    def retrieve(self, query: str, mode: str = 'auto') -> dict:
        if mode == 'llm_hybrid':
            l = self.local.retrieve(query)
            g = self.global_.retrieve(query)
            prompt = self._RERANK_PROMPT.format(
                query=query, local_ctx=l['context'], global_ctx=g['context'])
            try:
                merged = self._llm_call(prompt)
            except Exception as exc:
                merged = l['context'] + '\n\n' + g['context']
                print(f'[AdvancedDualRetriever] rerank failed: {exc}')
            return {'context': merged, 'retrieval_mode': 'llm_hybrid', 'local': l, 'global': g}
        return super().retrieve(query, mode)

    def answer(self, query: str, mode: str = 'auto') -> dict:
        ret = self.retrieve(query, mode)
        prompt = (
            'Answer the question using ONLY the provided knowledge graph context.\n'
            'If context is insufficient, say "Insufficient context."\n\n'
            f'Context:\n{ret["context"]}\n\nQuestion: {query}\n\nAnswer:'
        )
        try:
            answer = self._llm_call(prompt)
        except Exception as exc:
            answer = f'[Answer generation failed: {exc}]'
        return {'query': query, 'answer': answer,
                'context': ret['context'], 'retrieval_mode': ret['retrieval_mode']}


print('LLM_LocalRetriever, SemanticGlobalRetriever, AdvancedDualRetriever loaded.')


LLM_LocalRetriever, SemanticGlobalRetriever, AdvancedDualRetriever loaded.


## 21. . Conversational Retriever — Mistral-Nemo

Multi-turn conversation backed by knowledge graph retrieval.

**Flow per turn:**
1. **Query rewrite** — `mistral-nemo` condenses chat history + new question into a standalone retrieval query
2. **Graph retrieval** — `AdvancedDualRetriever` fetches context (`llm_hybrid` by default)
3. **Answer** — `mistral-nemo` generates a grounded response from context + history

Switch `conv_model='mistral-nemo'` to any model loaded via `ollama run <model>`.

In [21]:
class ConversationalRetriever:
    """Multi-turn conversational RAG using Mistral-Nemo with graph context.

    Usage:
        cr = ConversationalRetriever(adv_dual)
        cr.chat("What did Goldman Sachs acquire in 2022?")
        cr.chat("Who led that deal?")    # follow-up auto-uses history
        print(cr.history)
        cr.reset()
    """

    _REWRITE_PROMPT = (
        "Given the conversation history and a new user question, "
        "rewrite the question as a standalone search query capturing full context.\n"
        "Return ONLY the rewritten query — no explanation, no quotes.\n\n"
        "Conversation history:\n{history}\n\n"
        "New question: {question}\n\n"
        "Standalone query:"
    )

    _ANSWER_PROMPT = (
        "You are a financial knowledge assistant. "
        "Answer the user's question using the provided knowledge graph context and conversation history.\n"
        "Be concise and factual. If context is insufficient, say so explicitly.\n\n"
        "Knowledge Graph Context:\n{context}\n\n"
        "Conversation History:\n{history}\n\n"
        "User: {question}\n"
        "Assistant:"
    )

    def __init__(self, retriever: AdvancedDualRetriever,
                 conv_model: str = 'mistral-nemo',
                 retrieval_mode: str = 'llm_hybrid',
                 ollama_url: str = 'http://localhost:11434',
                 max_history_turns: int = 6):
        self.retriever       = retriever
        self.conv_model      = conv_model
        self.retrieval_mode  = retrieval_mode
        self.ollama_url      = ollama_url
        self.max_history_turns = max_history_turns
        self.history: List[Dict[str, str]] = []

    def _call(self, prompt: str, timeout: int = 90) -> str:
        resp = requests.post(
            f'{self.ollama_url}/api/generate',
            json={'model': self.conv_model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0.3, 'top_p': 0.9}},
            timeout=timeout
        )
        resp.raise_for_status()
        return resp.json()['response'].strip()

    def _format_history(self) -> str:
        if not self.history:
            return '(no prior turns)'
        lines = []
        for turn in self.history[-self.max_history_turns * 2:]:
            prefix = 'User' if turn['role'] == 'user' else 'Assistant'
            lines.append(f'{prefix}: {turn["content"]}')
        return '\n'.join(lines)

    def _rewrite_query(self, question: str) -> str:
        """Condense history + question into a standalone retrieval query via mistral-nemo."""
        if not self.history:
            return question
        prompt = self._REWRITE_PROMPT.format(
            history=self._format_history(), question=question)
        try:
            return self._call(prompt, timeout=30)
        except Exception as exc:
            print(f'[ConversationalRetriever] rewrite failed: {exc}')
            return question

    def chat(self, question: str, verbose: bool = True) -> str:
        """Send a message and return the assistant's response."""
        # 1. Rewrite for retrieval
        retrieval_query = self._rewrite_query(question)
        if verbose:
            print(f'  [retrieval query] {retrieval_query}')

        # 2. Retrieve graph context
        ret     = self.retriever.retrieve(retrieval_query, mode=self.retrieval_mode)
        context = ret['context']

        # 3. Generate answer with mistral-nemo
        prompt = self._ANSWER_PROMPT.format(
            context=context[:2500],
            history=self._format_history(),
            question=question,
        )
        try:
            answer = self._call(prompt)
        except Exception as exc:
            answer = f'[Generation failed: {exc}]'

        # 4. Update history
        self.history.append({'role': 'user',      'content': question})
        self.history.append({'role': 'assistant', 'content': answer})
        return answer

    def reset(self):
        """Clear conversation history."""
        self.history.clear()
        print('[ConversationalRetriever] History cleared.')

    def run_demo(self, questions: List[str]) -> None:
        """Run a canned multi-turn demo and print each exchange."""
        self.reset()
        print('=' * 65)
        print(f'ConversationalRetriever demo  model={self.conv_model}  retrieval={self.retrieval_mode}')
        print('=' * 65)
        for i, q in enumerate(questions, 1):
            print(f'\nTurn {i}  User: {q}')
            ans = self.chat(q, verbose=True)
            print(f'Assistant: {ans}')
        print('\n' + '=' * 65)


print('ConversationalRetriever (mistral-nemo) loaded.')


ConversationalRetriever (mistral-nemo) loaded.


In [ ]:
# -- Advanced Retrieval + Conversational Demo ----------------------------------
# Prerequisites:
#   ollama run llama3.2:3b     (entity extraction + reranking)
#   ollama run mistral-nemo    (conversational turns)

OLLAMA_AVAILABLE = globals().get('OLLAMA_AVAILABLE', True)

required = [
    'dual', 'NEL', 'LLM_LocalRetriever', 'SemanticGlobalRetriever',
    'AdvancedDualRetriever', 'ConversationalRetriever',
]
missing = [name for name in required if name not in globals()]

if not OLLAMA_AVAILABLE:
    print('[Skipped] Set OLLAMA_AVAILABLE=True and start llama3.2:3b + mistral-nemo via ollama')
elif missing:
    print(f"[Skipped] Run prerequisite cells first. Missing: {missing}")
else:
    # Reuse the graph built in the main demo (gb + dual are in scope from the demo cell)
    _G = dual.local.graph
    _nel = NEL(ticker_anchors={})

    _llm_local = LLM_LocalRetriever(_G, _nel, model='llama3.2:3b')
    _sem_global = SemanticGlobalRetriever(dual.global_.communities)
    adv_dual = AdvancedDualRetriever(_llm_local, _sem_global, model='llama3.2:3b')

    # -- Retrieval mode comparison ----------------------------------------------
    TEST_Q = 'What companies did Goldman Sachs deal with?'
    print(f'\nRetrieval mode comparison - "{TEST_Q}"')
    print('-' * 55)
    for mode in ('local', 'llm_local', 'semantic_global', 'llm_hybrid'):
        r = adv_dual.retrieve(TEST_Q, mode=mode)
        ctx_len = len(r['context'])
        print(f'  [{mode:16s}] {ctx_len:4d} chars  ', end='')
        # Show first line of context
        first = r['context'].split('\n')[0]
        print(first[:60])

    # -- Conversational multi-turn with mistral-nemo ----------------------------
    print()
    conv = ConversationalRetriever(
        adv_dual,
        conv_model='mistral-nemo',
        retrieval_mode='llm_hybrid',
    )
    CONV_QUESTIONS = [
        'What major acquisitions did Goldman Sachs make?',
        'Who was the CEO during those deals?',
        'What was the strategic rationale behind the Marcus acquisition?',
    ]
    conv.run_demo(CONV_QUESTIONS)

NameError: name 'dual' is not defined

## 22. . RAG Triad Evaluation (LLM-as-Judge)

Replaces F1-only evaluation with the full RAG quality triad:

| Metric | Definition |
|--------|-----------|
| **Faithfulness** | All answer claims grounded in the retrieved context |
| **Answer Relevance** | Answer addresses the question intent |
| **Context Precision** | Fraction of retrieved context relevant to the question |

LLM judge: Ollama (temp=0). Compatible with RAGAS / G-Eval frameworks.


In [24]:
class RAGTriadEvaluator:
    """Score RAG results with Faithfulness + Answer Relevance + Context Precision.
    Uses Ollama as judge (temperature=0). Each metric is 0.0–1.0.
    """

    _FAITH = (
        'Rate faithfulness of the answer to the context (0.0-1.0).\n'
        'Faithfulness = every claim in the answer is supported by the context.\n'
        '0.0 = hallucinated, 1.0 = fully grounded.\n\n'
        'Context: {context}\nAnswer: {answer}\n\n'
        'Output ONLY JSON: {{"score": <float>, "reason": "<one sentence>"}}'
    )
    _RELEV = (
        'Rate relevance of the answer to the question (0.0-1.0).\n'
        'Relevance = answer directly addresses the question.\n\n'
        'Question: {query}\nAnswer: {answer}\n\n'
        'Output ONLY JSON: {{"score": <float>, "reason": "<one sentence>"}}'
    )
    _PREC = (
        'Rate context precision (0.0-1.0).\n'
        'Precision = fraction of the context that is useful for answering the question.\n\n'
        'Question: {query}\nContext: {context}\n\n'
        'Output ONLY JSON: {{"score": <float>, "reason": "<one sentence>"}}'
    )

    def __init__(self, model: str = 'llama3.2:3b', dev_mode: bool = False,
                 ollama_url: str = 'http://localhost:11434'):
        self.model = 'llama3.2:1b' if dev_mode else model
        self.ollama_url = ollama_url

    def _judge(self, prompt: str) -> dict:
        try:
            resp = requests.post(f'{self.ollama_url}/api/generate',
                json={'model': self.model, 'prompt': prompt, 'stream': False,
                      'options': {'temperature': 0}},
                timeout=60)
            resp.raise_for_status()
            raw = resp.json()['response']
            hit = re.search(r'\{.*?\}', raw, re.DOTALL)
            if hit:
                return json.loads(hit.group())
        except Exception as exc:
            print(f'[RAGTriad judge] {exc}')
        return {'score': 0.0, 'reason': 'evaluation failed'}

    def score(self, result: dict) -> dict:
        query   = result.get('query', '')
        answer  = result.get('answer', '')
        context = result.get('context', '')[:2000]

        faith = self._judge(self._FAITH.format(context=context, answer=answer))
        relev = self._judge(self._RELEV.format(query=query, answer=answer))
        prec  = self._judge(self._PREC.format(query=query, context=context))

        f, r, p = float(faith['score']), float(relev['score']), float(prec['score'])
        return {
            'query':              query,
            'retrieval_mode':     result.get('retrieval_mode', '?'),
            'faithfulness':       round(f, 3),
            'answer_relevance':   round(r, 3),
            'context_precision':  round(p, 3),
            'rag_score':          round((f + r + p) / 3, 3),
            'faith_reason':       faith.get('reason', ''),
            'relev_reason':       relev.get('reason', ''),
            'prec_reason':        prec.get('reason', ''),
        }

    def evaluate_batch(self, results: List[dict]) -> List[dict]:
        out = []
        for r in results:
            print(f"  Scoring: {r.get('query','')[:60]}...")
            out.append(self.score(r))
        return out

    def summary(self, scores: List[dict]) -> dict:
        if not scores: return {}
        keys = ['faithfulness', 'answer_relevance', 'context_precision', 'rag_score']
        return {k: round(sum(s[k] for s in scores) / len(scores), 3) for k in keys}


# ── Baseline stubs (vector RAG comparison) ────────────────────────────────────
class VectorRAG_Baseline:
    """Naive cosine-similarity retrieval stub — no graph, pure embedding.
    Intended as ablation baseline. Requires sentence-transformers.
    """
    def answer(self, query: str, corpus: List[str]) -> dict:
        try:
            from sentence_transformers import SentenceTransformer, util
            _m = SentenceTransformer('all-MiniLM-L6-v2')
            q_emb = _m.encode(query, convert_to_tensor=True)
            c_emb = _m.encode(corpus, convert_to_tensor=True)
            scores = util.cos_sim(q_emb, c_emb)[0]
            top_idx = int(scores.argmax())
            context = corpus[top_idx]
        except ImportError:
            context = 'Vector baseline unavailable (sentence-transformers not installed).'
        return {'query': query, 'answer': context[:400],
                'context': context, 'retrieval_mode': 'vector_rag'}


print('RAGTriadEvaluator + VectorRAG_Baseline loaded.')


RAGTriadEvaluator + VectorRAG_Baseline loaded.


## 23. Pipeline Orchestrator

`FinBERTGraphPipeline` is the unified end-to-end orchestrator.

- **NER**: `extract_entities()` — FinBERT → EARE → SSHG → GNN → BIO
- **Linking**: `NEL` with LEI/GID canonical IDs
- **Temporal**: point-in-time validity enforcement
- **RE primary**: `classify_relation()` — bilinear head + pattern prior
- **RE supplementary**: `Ollama_RE` — only for low-confidence pairs (optional)
- **Graph**: versioned append-only `GraphBuilder`
- **RAG**: `DualRetriever` after `CommunityDetector` + `CommunitySummarizer`

In [25]:
class FinBERTGraphPipeline:
    """Unified orchestrator: FinBERT extraction → hierarchical GraphRAG."""

    def __init__(self, chunk_size: int = 512, ticker_linking: bool = True,
                 ollama_url: str = None, dev_mode: bool = False,
                 use_supplementary_re: bool = False,
                 supplementary_conf_threshold: float = 0.35):
        self._pp          = Preprocessor(chunk_size=chunk_size, ticker_linking=ticker_linking)
        self._tl          = TemporalLayer()
        self._prov        = ProvenanceEnforcer()
        self.ollama_url   = ollama_url or CFG["ollama_url"]
        self.dev_mode     = dev_mode
        self.use_sup_re   = use_supplementary_re
        self.sup_threshold = supplementary_conf_threshold

    def _fallback_link(self, text: str) -> Link:
        return Link(entity=text,
                    canonical_id=f"LOCAL:{hashlib.md5(text.lower().encode()).hexdigest()[:8]}",
                    confidence=0.0, valid_time={'start': None, 'end': None})

    def ingest_doc(self, doc_id: str, text: str, date: Optional[str] = None,
                   nel: NEL = None, gb: GraphBuilder = None) -> List[dict]:
        """
        Process one document through the full pipeline.

        Steps:
          1. Preprocess → chunks + ticker anchors
          2. For each chunk:
             a. FinBERT/GNN NER → Entity list
             b. NEL → Link list
             c. Temporal date inference
             d. Bilinear RE (primary) + optional Ollama RE (supplementary)
             e. Provenance enforcement
             f. Graph builder ingestion
        """
        chunks, ticker_anchors = self._pp.process(doc_id, text)
        if nel: nel.ticker_anchors.update(ticker_anchors)

        results = []
        for chunk in chunks:
            # ── a. NER ──────────────────────────────────────────────────────
            chunk.entities = extract_entities(chunk.text)

            # ── b. NEL ──────────────────────────────────────────────────────
            if nel and chunk.entities:
                chunk.links = nel.link_all(chunk.entities, context=chunk.text)
            e2l = {lnk.entity: lnk for lnk in chunk.links}

            # ── c. Temporal ─────────────────────────────────────────────────
            rels = []

            # ── d. RE: bilinear primary ──────────────────────────────────────
            ents = chunk.entities
            for i, ei in enumerate(ents):
                for ej in ents[i+1:]:
                    rel_name, conf = classify_relation(
                        ei.text, ej.text, ei.type, ej.type, chunk.text)
                    if rel_name == "NO_RELATION":
                        continue
                    t = self._tl.infer_relation_time(chunk.text, date)
                    prov = Provenance(source_chunk_id=chunk.chunk_id,
                                      raw_sentence=chunk.text[:300])
                    rels.append(Relation(subj=ei.text, rel=rel_name, obj=ej.text,
                                        time=t, provenance=prov, confidence=conf))

            # ── d2. RE: Ollama supplementary (optional) ───────────────────────
            if self.use_sup_re and chunk.entities:
                try:
                    sup_re = get_supplementary_re(self.ollama_url, self.dev_mode)
                    sup_rels = sup_re.extract(chunk.text, chunk.entities)
                    # Only keep if not already covered by primary RE
                    existing = {(r.subj.lower(), r.obj.lower()) for r in rels}
                    for r in sup_rels:
                        if (r.subj.lower(), r.obj.lower()) not in existing:
                            rels.append(r)
                except Exception as ex:
                    pass  # Ollama unavailable — primary RE is sufficient

            # ── e. Provenance enforcement ────────────────────────────────────
            for r in rels:
                self._prov.enforce(r, chunk.chunk_id, chunk.text[:300])
            rels = self._prov.filter_valid(rels)
            chunk.relations = rels

            # ── f. Graph builder ─────────────────────────────────────────────
            if gb:
                for r in rels:
                    ls = e2l.get(r.subj, self._fallback_link(r.subj))
                    lo = e2l.get(r.obj,  self._fallback_link(r.obj))
                    if not self._tl.is_valid(r.time, ls.valid_time) or \
                       not self._tl.is_valid(r.time, lo.valid_time):
                        continue
                    gb.add_relation(doc_id, ls, lo, r)

            results.append(chunk.to_dict())
        return results

    def build_rag(self, gb: GraphBuilder, nel: NEL = None,
                  summarize: bool = True) -> DualRetriever:
        """Build DualRetriever from a populated GraphBuilder."""
        G   = gb.get_graph()
        det = CommunityDetector()
        l0  = det.detect(G)
        com = det.build_hierarchy(G, l0)
        if summarize:
            summ = CommunitySummarizer(dev_mode=self.dev_mode, ollama_url=self.ollama_url)
            com  = summ.summarize_all(com)
        nel_obj = nel or NEL(ticker_anchors={})
        return DualRetriever(LocalRetriever(G, nel_obj), GlobalRetriever(com),
                             model='llama3.2:1b' if self.dev_mode else 'llama3.2:3b',
                             ollama_url=self.ollama_url)


print("FinBERTGraphPipeline ready.")

FinBERTGraphPipeline ready.


## 24. . Demo — Hierarchical GraphRAG on Sample Bloomberg Docs

Full pipeline run: extraction → graph → Leiden communities → LLM summaries → dual retrieval → RAG triad scoring.

> **Dev mode** (`dev_mode=True`): uses `llama3.2:1b`, skips Ollama calls if unavailable.  
> Set `OLLAMA_AVAILABLE = True` once `ollama run llama3.2:3b` is running.


In [ ]:
def test_provenance_invariant(results: List[dict]) -> List[str]:
    errors = []
    for r in results:
        for rel in r.get('relations', []):
            prov = rel.get('provenance', {})
            if not prov.get('source_chunk_id'):
                errors.append(f'FAIL missing source_chunk_id: {rel["subj"]}->{rel["obj"]}')
            if not prov.get('raw_sentence'):
                errors.append(f'FAIL missing raw_sentence: {rel["subj"]}->{rel["obj"]}')
        for lnk in r.get('links', []):
            if not lnk.get('canonical_id'):
                errors.append(f'FAIL missing canonical_id: {lnk}')
    return errors


def test_append_only() -> Tuple[bool, str]:
    _gb = GraphBuilder()
    ls = Link('GS', 'LEI:GS001', 0.9, {'start': '2020-01-01', 'end': None})
    lo = Link('Marcus', 'LOCAL:marc', 0.5, {'start': None, 'end': None})
    r1 = Relation('GS', 'acquired', 'Marcus', {'start': '2022-01-01', 'end': None},
                  Provenance('c1', 'GS acquired Marcus in 2022.'), 0.8)
    r2 = Relation('GS', 'merger_cancelled', 'Marcus', {'start': '2023-01-01', 'end': None},
                  Provenance('c2', 'GS merger cancelled.'), 0.7)
    _gb.add_relation('doc', ls, lo, r1)
    _gb.add_relation('doc', ls, lo, r2)
    n = _gb.get_graph().number_of_edges()
    return n == 2, f'expected 2 edges, got {n}'


def test_community_schema(communities: Dict) -> List[str]:
    errors = []
    for cid, node in communities.items():
        if not isinstance(node.cluster_id, str) or not node.cluster_id:
            errors.append(f'FAIL missing cluster_id for key {cid}')
        if node.level not in (0, 1, 2):
            errors.append(f'FAIL invalid level {node.level} for {node.cluster_id}')
        if not isinstance(node.entities, list):
            errors.append(f'FAIL entities not a list for {node.cluster_id}')
    return errors


def test_router(nel_obj: NEL) -> List[str]:
    errors = []
    _local = LocalRetriever(nx.MultiDiGraph(), nel_obj)
    _global = GlobalRetriever({})
    _dual = DualRetriever(_local, _global)
    entity_q = 'What did Goldman Sachs acquire?'
    thematic_q = 'What sectors saw consolidation?'
    if not _dual._has_entities(entity_q):
        errors.append(f'FAIL router should detect entities in: {entity_q}')
    if _dual._has_entities(thematic_q):
        errors.append(f'FAIL router should not detect entities in: {thematic_q}')
    return errors


print('=' * 55)
print('Invariant Tests')
print('=' * 55)

# Provenance test corpus built with currently defined pipeline components.
all_results_flat = []
if 'SAMPLE_DOCS' in globals() and 'FinBERTGraphPipeline' in globals() and 'NEL' in globals() and 'GraphBuilder' in globals():
    _pp2 = FinBERTGraphPipeline(chunk_size=256, dev_mode=True)
    _nel2 = NEL(ticker_anchors={}, strategy='fuzzy_flat')
    _gb2 = GraphBuilder()
    for doc in SAMPLE_DOCS:
        all_results_flat.extend(_pp2.ingest_doc(
            doc_id=doc['doc_id'],
            text=doc['text'],
            date=doc['date'],
            nel=_nel2,
            gb=_gb2,
        ))
else:
    print('[WARN] Skipping provenance corpus build: run demo setup cells first.')

prov_errs = test_provenance_invariant(all_results_flat)
print(f'[{"PASS" if not prov_errs else "FAIL"}] Provenance invariant'
      + (f' - {prov_errs[:2]}' if prov_errs else ''))

ao_ok, ao_msg = test_append_only()
print(f'[{"PASS" if ao_ok else "FAIL"}] Append-only - {ao_msg}')

if 'communities' in globals():
    comm_errs = test_community_schema(communities)
    print(f'[{"PASS" if not comm_errs else "FAIL"}] Community schema'
          + (f' - {comm_errs[:2]}' if comm_errs else ''))
else:
    print('[WARN] Skipping community schema test: run community detection cell first.')

if 'nel' in globals():
    router_errs = test_router(nel)
else:
    router_errs = test_router(NEL(ticker_anchors={}, strategy='fuzzy_flat'))
print(f'[{"PASS" if not router_errs else "FAIL"}] Retrieval router'
      + (f' - {router_errs}' if router_errs else ''))

Hierarchical GraphRAG — Demo

[1] Extracting entities + relations (FinBERT pipeline)...
  bloomberg_001: [('Goldman', 'INVESTED_IN', 'Goldman Sachs'), ('Goldman', 'INVESTED_IN', 'Marcus Investments'), ('US', 'INVESTED_IN', 'Goldman Sachs'), ('US', 'INVESTED_IN', 'Marcus Investments'), ('Goldman Sachs', 'ACQUIRED', 'Marcus Investments')]
  bloomberg_002: [('and', 'INVESTED_IN', '.'), ('and', 'INVESTED_IN', 'Microsoft'), (')', 'INVESTED_IN', '.'), (')', 'INVESTED_IN', 'Microsoft'), ('.', 'INVESTED_IN', 'Microsoft')]
  bloomberg_003: [('JP Morgan Chase', 'ACQUIRED', 'First Republic Bank'), ('JP Morgan Chase', 'REPORTED_REVENUE', '10.6B.'), ('First Republic Bank', 'REPORTED_REVENUE', '10.6B.')]

[2] Graph stats:
  nodes: 10
  edges: 13
  provenance_pct: 100.0
  timestamp_pct: 100.0
  conflict_groups: 0

[3] Community detection (Leiden / networkx fallback)...


NameError: name 'CommunityNode' is not defined

## 25. . Continuous Testing

In [ ]:
def test_provenance_invariant(results: List[dict]) -> List[str]:
    errors = []
    for r in results:
        for rel in r.get('relations', []):
            prov = rel.get('provenance', {})
            if not prov.get('source_chunk_id'):
                errors.append(f'FAIL missing source_chunk_id: {rel["subj"]}→{rel["obj"]}')
            if not prov.get('raw_sentence'):
                errors.append(f'FAIL missing raw_sentence: {rel["subj"]}→{rel["obj"]}')
        for lnk in r.get('links', []):
            if not lnk.get('canonical_id'):
                errors.append(f'FAIL missing canonical_id: {lnk}')
    return errors


def test_append_only() -> Tuple[bool, str]:
    _gb = GraphBuilder()
    ls = Link('GS', 'LEI:GS001', 0.9, {'start': '2020-01-01', 'end': None})
    lo = Link('Marcus', 'LOCAL:marc', 0.5, {'start': None, 'end': None})
    r1 = Relation('GS', 'acquired', 'Marcus', {'start': '2022-01-01', 'end': None},
                  Provenance('c1', 'GS acquired Marcus in 2022.'), 0.8)
    r2 = Relation('GS', 'merger_cancelled', 'Marcus', {'start': '2023-01-01', 'end': None},
                  Provenance('c2', 'GS merger cancelled.'), 0.7)
    _gb.add_relation('doc', ls, lo, r1)
    _gb.add_relation('doc', ls, lo, r2)
    n = _gb.get_graph().number_of_edges()
    return n == 2, f'expected 2 edges, got {n}'


def test_community_schema(communities: Dict) -> List[str]:
    errors = []
    for cid, node in communities.items():
        if not isinstance(node.cluster_id, str) or not node.cluster_id:
            errors.append(f'FAIL missing cluster_id for key {cid}')
        if node.level not in (0, 1, 2):
            errors.append(f'FAIL invalid level {node.level} for {node.cluster_id}')
        if not isinstance(node.entities, list):
            errors.append(f'FAIL entities not a list for {node.cluster_id}')
    return errors


def test_router(nel: NEL) -> List[str]:
    errors = []
    _local = LocalRetriever(nx.MultiDiGraph(), nel)
    _global = GlobalRetriever({})
    _dual = DualRetriever(_local, _global)
    entity_q  = 'What did Goldman Sachs acquire?'
    thematic_q = 'What sectors saw consolidation?'
    if not _dual._has_entities(entity_q):
        errors.append(f'FAIL router should detect entities in: {entity_q}')
    if _dual._has_entities(thematic_q):
        errors.append(f'FAIL router should not detect entities in: {thematic_q}')
    return errors


print('=' * 55)
print('Invariant Tests')
print('=' * 55)

# Provenance
all_results_flat = []
for doc in SAMPLE_DOCS:
    _pp2 = HierarchicalPipeline(chunk_size=256)
    _nel2 = NEL(ticker_anchors={}, strategy='fuzzy_flat')
    _gb2 = GraphBuilder()
    all_results_flat.extend(_pp2.process_document(
        doc_id=doc['doc_id'], text=doc['text'], date=doc['date'],
        ner=SpacyNER(), nel=_nel2, re_extractor=Snowball_RE(), gb=_gb2))

prov_errs = test_provenance_invariant(all_results_flat)
print(f'[{"PASS" if not prov_errs else "FAIL"}] Provenance invariant'
      + (f' — {prov_errs[:2]}' if prov_errs else ''))

ao_ok, ao_msg = test_append_only()
print(f'[{"PASS" if ao_ok else "FAIL"}] Append-only — {ao_msg}')

comm_errs = test_community_schema(communities)
print(f'[{"PASS" if not comm_errs else "FAIL"}] Community schema'
      + (f' — {comm_errs[:2]}' if comm_errs else ''))

router_errs = test_router(nel)
print(f'[{"PASS" if not router_errs else "FAIL"}] Retrieval router'
      + (f' — {router_errs}' if router_errs else ''))


## End-to-End Flow

```
Bloomberg Financial News 120K
  │
  ▼  [1] Preprocessing        Preprocessor: ticker anchoring, normalise, chunk(512)
  │
  ▼  [2] FinBERT Encoder       ProsusAI/finbert → H ∈ ℝⁿˣ⁷⁶⁸
  ▼  [3] EARE                  entity-aware gating → Ĥ ∈ ℝⁿˣ⁷⁶⁸
  ▼  [4] SSHG                  dep arcs (λ₁=1.0) + cosine window (λ₂=0.5) → G_SSHG
  ▼  [5] GNN (L=2)             weighted message passing → H_GNN ∈ ℝⁿˣ²⁵⁶
  ▼  [6] BIO Classifier        MLP → BIO tags → Entity spans, merged with spaCy
  │
  ▼  [7] NEL                   ticker → LEI/GID alias → rapidfuzz → LOCAL fingerprint
  ▼  [8] Temporal              date extraction, PIT validity [{start, end}]
  │
  ▼  [9] Relation Extraction
  │      Primary:   pattern-rule prior (conf=0.85) → bilinear head blend (α=0.60)
  │      Fallback:  Ollama LLM joint extractor (mistral, temp=0) — low-conf pairs
  │
  ▼  [10] Provenance           source_chunk_id + raw_sentence enforced — hard invariant
  ▼  [11] Graph Builder        append-only, MultiDiGraph, canonical_id node keys
  ▼  [12] Community Detection  Leiden → 3-level hierarchy (L0/L1/L2)
  ▼  [13] Community Summaries  Ollama, grounded in graph edges only
  │
  ▼  [14] Retrieval
  │      Local:   entity → NEL → subgraph BFS → context
  │      Global:  keyword / semantic → cluster summary
  │      Hybrid:  LLM re-ranks local + global (llama3.2:3b)
  │      Conv:    Mistral-Nemo multi-turn + query rewrite
  │
  ▼  [15] RAG Triad Evaluation Faithfulness · Answer Relevance · Context Precision
```

**Hard invariants**
- No relation without `provenance.source_chunk_id` + `provenance.raw_sentence`
- No entity without `canonical_id` (LEI/GID or LOCAL fingerprint)
- No relation without `time` dict (explicit `null` accepted)
- Append-only graph — parallel edges, no overwrite
- All summaries derived from graph edges only — no external knowledge injection
- Deterministic LLM calls (temperature = 0)